In [46]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [3]:
import pandas as pd

data = pd.read_csv("./ChnSentiCorp_htl_all.csv")
data

,label,review
0,1,"距离川沙公路较近,但是公交指示不对,如果是""蔡陆线""的话,会非常麻烦.建议用别的路线.房间较..."
1,1,商务大床房，房间很大，床有2M宽，整体感觉经济实惠不错!
2,1,早餐太差，无论去多少人，那边也不加食品的。酒店应该重视一下这个问题了。房间本身很好。
3,1,宾馆在小街道上，不大好找，但还好北京热心同胞很多~宾馆设施跟介绍的差不多，房间很小，确实挺小...
4,1,"CBD中心,周围没什么店铺,说5星有点勉强.不知道为什么卫生间没有电吹风"
...,...,...
7761,0,尼斯酒店的几大特点：噪音大、环境差、配置低、服务效率低。如：1、隔壁歌厅的声音闹至午夜3点许...
7762,0,盐城来了很多次，第一次住盐阜宾馆，我的确很失望整个墙壁黑咕隆咚的，好像被烟熏过一样家具非常的...
7763,0,看照片觉得还挺不错的，又是4星级的，但入住以后除了后悔没有别的，房间挺大但空空的，早餐是有但...
7764,0,我们去盐城的时候那里的最低气温只有4度，晚上冷得要死，居然还不开空调，投诉到酒店客房部，得到...


In [6]:
data = data.dropna()
type(data)

pandas.core.frame.DataFrame

# 构造数据集

In [12]:
from torch.utils.data import Dataset

class MyDataset(Dataset):
    def __init__(self):
        super().__init__()
        self.data = pd.read_csv("./ChnSentiCorp_htl_all.csv").dropna()
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, index):
        return self.data.iloc[index]["label"], self.data.iloc[index]["review"]

In [13]:
dataset = MyDataset()
for i in range(5):
    print(dataset[i])

(np.int64(1), '距离川沙公路较近,但是公交指示不对,如果是"蔡陆线"的话,会非常麻烦.建议用别的路线.房间较为简单.')
(np.int64(1), '商务大床房，房间很大，床有2M宽，整体感觉经济实惠不错!')
(np.int64(1), '早餐太差，无论去多少人，那边也不加食品的。酒店应该重视一下这个问题了。房间本身很好。')
(np.int64(1), '宾馆在小街道上，不大好找，但还好北京热心同胞很多~宾馆设施跟介绍的差不多，房间很小，确实挺小，但加上低价位因素，还是无超所值的；环境不错，就在小胡同内，安静整洁，暖气好足-_-||。。。呵还有一大优势就是从宾馆出发，步行不到十分钟就可以到梅兰芳故居等等，京味小胡同，北海距离好近呢。总之，不错。推荐给节约消费的自助游朋友~比较划算，附近特色小吃很多~')
(np.int64(1), 'CBD中心,周围没什么店铺,说5星有点勉强.不知道为什么卫生间没有电吹风')


In [20]:
# 划分下数据集， 9成训练数据， 1成测试数据

from torch.utils.data import random_split

trainset, validset = random_split(dataset, lengths=[0.9, 0.1])
len(trainset), len(validset)

(6989, 776)

# 创建DataLoader

In [36]:
import torch
tokenizer = AutoTokenizer.from_pretrained("./rbt3")

def collate_func(batch):
    reviews, labels = [], []
    for label, review in batch:
        reviews.append(review)
        labels.append(label)
    inputs = tokenizer(reviews, padding="max_length", truncation=True, max_length=128, return_tensors="pt")
    inputs["labels"] = torch.tensor(labels)
    return inputs
        
        



In [37]:
from torch.utils.data import DataLoader

trainloader = DataLoader(trainset, batch_size=32, shuffle=True, collate_fn=collate_func)
validloader = DataLoader(validset, batch_size=64, shuffle=False, collate_fn=collate_func)

In [44]:
for k, v in next(iter(trainloader)).items():
    print(k, v.shape)
    
    

input_ids torch.Size([32, 128])
token_type_ids torch.Size([32, 128])
attention_mask torch.Size([32, 128])
labels torch.Size([32])


# 创建模型和优化器

In [47]:
from torch.optim import Adam
model = AutoModelForSequenceClassification.from_pretrained("./rbt3")
if torch.cuda.is_available():
    model = model.cuda()
optimizer = Adam(model.parameters(), lr=2e-5)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at ./rbt3 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


# 训练和验证

In [59]:
def evaluate():
    model.eval()
    acc_num = 0
    with torch.inference_mode():
        for batch in validloader:
            if torch.cuda.is_available():
                batch = {k:v.cuda() for k, v in batch.items()}
            outputs = model(**batch)
            pred = outputs.logits.argmax(dim=-1)
            acc_num += (pred == batch["labels"]).sum()
            
    return acc_num / len(validset)

In [60]:
def train(epoch=3, log_step = 100):
    global_step = 0
    model.train()
    for ep in range(epoch):
        for batch in trainloader:
            if torch.cuda.is_available():
                batch = {k:v.cuda() for k, v in batch.items()}
            optimizer.zero_grad()
            output = model(**batch)
            output.loss.backward()
            optimizer.step()
            if global_step % log_step == 0:
                print(f"Epoch: {ep}, Step: {global_step}, Loss: {output.loss.item()}")
            global_step += 1
        acc = evaluate()
        print(f"Epoch: {ep}, Accuracy: {acc}")  

# 训练

In [61]:
train()

Epoch: 0, Step: 0, Loss: 0.13210725784301758
Epoch: 0, Step: 100, Loss: 0.1309441775083542
Epoch: 0, Step: 200, Loss: 0.1962360292673111
Epoch: 0, Accuracy: 0.8865979313850403
Epoch: 1, Step: 300, Loss: 0.03677717596292496
Epoch: 1, Step: 400, Loss: 0.052469611167907715
Epoch: 1, Accuracy: 0.8801546096801758
Epoch: 2, Step: 500, Loss: 0.04419780522584915
Epoch: 2, Step: 600, Loss: 0.18038703501224518
Epoch: 2, Accuracy: 0.875


# 

# 模型预测

In [65]:
sen = "我觉得这家酒店不错，饭很好吃！"
id2label = {0:"负面", 1:"正面"}

with torch.inference_mode():
    inputs = tokenizer(sen, return_tensors="pt", truncation=True, max_length=128)
    inputs = {k:v.cuda() for k, v in inputs.items()} if torch.cuda.is_available() else inputs
    ouputs = model(**inputs)
    pred = ouputs.logits.argmax(dim=-1)
    print(f"输入：{sen}\n模型预测结果:{id2label[pred.item()]}")

输入：我觉得这家酒店不错，饭很好吃！
模型预测结果:正面


# 使用pipeline来整合上述所有操作 

In [66]:
from transformers import pipeline
model.config.id2label = id2label
pipe = pipeline("text-classification", model=model, tokenizer=tokenizer, device=0 if torch.cuda.is_available() else "cpu")



Device set to use cuda:0


In [67]:
pipe(sen)

[{'label': '正面', 'score': 0.9963328838348389}]